# 4. BIDCell — Deep Learning Segmentation Workflow

BIDCell requires more manual intervention than ProSeg/Baysor/Cellpose because:
1. It uses its own YAML config for model architecture & training
2. It outputs a `.tif` label mask that needs resizing to match DAPI dimensions
3. The resized mask then needs to be imported via Xenium Ranger for Explorer

This notebook walks through:
- Generating and editing the BIDCell config
- Inspecting outputs after the SLURM job
- Post-processing the label mask for Explorer import

In [ ]:
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

In [ ]:
CONFIG_PATH = "../config/pipeline_config.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

SAMPLE_ID = cfg["sample"]["id"]
BIDCELL_DIR = Path(cfg["paths"]["base_output_dir"]) / SAMPLE_ID / "bidcell"
RAW_DATA = cfg["sample"]["raw_data_path"]

print(f"BIDCell output dir: {BIDCELL_DIR}")

## Step 1: Generate BIDCell config template
Run this **before** submitting the SLURM job if you need to customize.

In [ ]:
from bidcell import BIDCellModel

# Generate platform-specific config
platform = cfg["sample"]["platform"]  # "xenium", "cosmx", etc.
BIDCellModel.get_example_config(platform)
print(f"Generated: {platform}_example_config.yaml")
print("Edit this file to set data paths, then move to the BIDCell output dir.")

In [ ]:
# View the generated config
config_file = f"{platform}_example_config.yaml"
if os.path.exists(config_file):
    with open(config_file) as f:
        print(f.read())

## Step 2: Inspect BIDCell outputs (after SLURM job)
BIDCell writes label masks as `.tif` files.

In [ ]:
# List output files
if BIDCELL_DIR.exists():
    for f in sorted(BIDCELL_DIR.rglob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / (1024**2)
            print(f"  {f.relative_to(BIDCELL_DIR)}  ({size_mb:.1f} MB)")
else:
    print(f"BIDCell output dir not found: {BIDCELL_DIR}")
    print("Submit the BIDCell SLURM job first.")

In [ ]:
# Load and visualize the label mask
import tifffile

tif_files = list(BIDCELL_DIR.rglob("*.tif")) + list(BIDCELL_DIR.rglob("*.tiff"))
if tif_files:
    mask_path = tif_files[0]
    cells = tifffile.imread(str(mask_path))
    print(f"Label mask: {mask_path.name}")
    print(f"  Shape: {cells.shape}")
    print(f"  Dtype: {cells.dtype}")
    print(f"  Unique labels: {len(np.unique(cells))} (including background)")
    print(f"  Cell count: {len(np.unique(cells)) - 1}")
    
    # Quick view
    fig, ax = plt.subplots(figsize=(10, 10))
    # Show a crop
    crop = cells[:2000, :2000] if cells.shape[0] > 2000 else cells
    ax.imshow(crop > 0, cmap="gray")
    ax.set_title(f"BIDCell mask (crop) — {len(np.unique(crop)) - 1} cells")
    plt.show()
else:
    print("No .tif files found in BIDCell output.")

## Step 3: Resize mask to match DAPI dimensions
BIDCell may output at a different resolution than the DAPI image. Resize with nearest-neighbor interpolation to preserve integer labels.

In [ ]:
import cv2
from spatialdata_io import xenium

# Load the raw data to get DAPI dimensions
sdata = xenium(RAW_DATA, cells_as_circles=True)

if "morphology_focus" in sdata.images:
    dapi = sdata.images["morphology_focus"]
    # Get spatial dimensions (last 2 for CYX format)
    h_dapi, w_dapi = dapi.shape[-2], dapi.shape[-1]
    print(f"DAPI dimensions: {h_dapi} × {w_dapi}")
    
    if tif_files:
        print(f"Mask dimensions: {cells.shape}")
        
        if cells.shape != (h_dapi, w_dapi):
            print("Resizing mask to match DAPI...")
            cells_resized = cv2.resize(
                cells.astype('float32'),
                (w_dapi, h_dapi),
                interpolation=cv2.INTER_NEAREST
            ).astype(np.uint32)
            
            out_path = BIDCELL_DIR / f"{SAMPLE_ID}_bidcell_labels_resized.tif"
            tifffile.imwrite(str(out_path), cells_resized)
            print(f"Saved resized mask: {out_path}")
            print(f"  Shape: {cells_resized.shape}")
            print(f"  Cells: {len(np.unique(cells_resized)) - 1}")
        else:
            print("Mask already matches DAPI dimensions.")
else:
    print("morphology_focus image not found")

## Step 4: Import into Xenium Explorer

Use Xenium Ranger to import the resized label mask:

```bash
xeniumranger import-segmentation \
    --id=bidcell_import \
    --xenium-bundle=/path/to/xenium/output \
    --cells=segmentation_results/{SAMPLE_ID}/bidcell/{SAMPLE_ID}_bidcell_labels_resized.tif
```

This produces a bundle that Xenium Explorer can open.